# Import libraries

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from datasets import load_dataset
import pandas as pd

/Users/fcalado/Desktop/anti-trans-legislation/training/heritage/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Sentiment Analysis

By default, the 🤗 Transformers library text-classification pipeline uses the [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) model. This model is based on a distilled, uncased version of [BERT](https://huggingface.co/bert-base-uncased) that has been fine-tuned on the [Stanford Sentiment Treebank 2](https://huggingface.co/datasets/sst2) (SST-2) dataset. The SST-2 dataset is a binary classification dataset for training models to learn the sentiment of words, phrases, and sentences. It contains 215,154 unique manually labeled texts of varying lengths. The model card describes SST-2:

> The corpus is based on the dataset introduced by Pang and Lee (2005) and consists of 11,855 single sentences extracted from movie reviews. It was parsed with the Stanford parser and includes a total of 215,154 unique phrases from those parse trees, each annotated by 3 human judges.

In [15]:
# Sentiment Analysis
classifier = pipeline("text-classification", model="SamLowe/roberta-base-go_emotions")

Device set to use mps:0


In [16]:
# Define a task function
def classify_sentiment(prompt):
        output = classifier(prompt)
        return output

In [ ]:
# Pass a prompt into the task function
classify_sentiment(ds['train']['text'][10])

[{'label': 'neutral', 'score': 0.6804832816123962}]

In [19]:
for i in ds['train']['text'][:10]:
    print(i)
    print(classify_sentiment(i))

The weaponization of the U.S. Department of Justice against purported enemies of the Biden-Harris administration is nothing new.
[{'label': 'neutral', 'score': 0.7223581671714783}]
The Department’s persecution of pro-life advocates, praying grandmothers, and fathers of young children under the Freedom of Access to Clinic Entrances Act (while routinely ignoring arson and vandalism at crisis pregnancy centers chargeable under the same law) is hard to deny.
[{'label': 'neutral', 'score': 0.7027313113212585}]
But the Department’s meritless criminal prosecution of Dr. Eithan Haim, a whistleblowing physician who exposed covert gender medicine procedures inflicted on minors at Texas Children’s Hospital—performed in violation of state law—is an outrageous abuse of its law enforcement authority.
[{'label': 'annoyance', 'score': 0.43028414249420166}]
The principal wrongdoer in Dr. Haim’s case—leading the charge in his criminal prosecution for alleged violation of the Health Insurance Portability

## Question Answering

The Question Answering pipeline has two required parameters: 

* `question` The question being asked
* `context` The source material that should be used to answer this question

In [25]:
# Question Answering
reader = pipeline("question-answering")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


In [26]:
# Define a task function
def answer_question(question, context):
    output = reader(question=question, context=context)
    return pd.DataFrame([output])

In [45]:
# grab the first 50 sentences as "context"

input_sentences = ds['train']['text'][:50]
context = ''.join(input_sentences)

In [ ]:
# Pass the prompt into the task function

question = "what's going on?"
answer = answer_question(question, context)

In [49]:
# it returns a dataframe

answer

,score,start,end,answer
0,0.029369,128,178,The Department’s persecution of pro-life advoc...


In [47]:
answer['answer'][0]

'The Department’s persecution of pro-life advocates'

## Summarization

The `clean_up_tokenization_spaces` parameter removes extraneous spaces created through the detokenization process. If tokenization breaks up a string into separate tokens, then detokenization joins together a series of tokens into a string.

In [2]:
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")
pipe = pipeline("summarization", model="facebook/bart-large-cnn")

Device set to use mps:0


In [3]:
df = pd.read_json('cleaned.json', lines=True)
df.head()

,title,author,date,url,takeaways,text
0,Biden-Harris DOJ Prosecutor Targets Trans Whis...,"[Hans A. von Spakovsky, Sarah Parshall Perry]",2024-11-07,https://www.heritage.org/courts/commentary/bid...,[The weaponization of the U.S. Department of J...,The weaponization of the U.S. Department of Ju...
1,Wisconsin Public Schools’ Gender Policies Shut...,"[Thomas Jipping, Sarah Parshall Perry]",2024-10-23,https://www.heritage.org/gender/commentary/wis...,[Schools are generally in charge of matters su...,The Supreme Court has said that parents’ right...
2,United States v. Skrmetti: Oral Arguments Indi...,[Sarah Parshall Perry],2024-12-13,https://www.heritage.org/gender/commentary/uni...,[Tennessee is one of 26 states that have enact...,"On Wednesday, December 3, the U.S. Supreme Cou..."
3,Trump’s Transgender Orders Are Well Within Exe...,"[Paul J. Larkin, Sarah Parshall Perry]",2025-02-19,https://www.heritage.org/gender/commentary/tru...,[They claim the president’s directive prohibit...,"In two short weeks, President Donald Trump has..."
4,Follow the Law,[Sarah Parshall Perry],2025-02-27,https://www.heritage.org/gender/commentary/fol...,[Hurson’s decision halts the directives of Pre...,It’s difficult to envision a more dizzying exe...


In [4]:
df['text'][0]

'The weaponization of the U.S. Department of Justice against purported enemies of the Biden-Harris administration is nothing new. The Department’s persecution of pro-life advocates, praying grandmothers, and fathers of young children under the Freedom of Access to Clinic Entrances Act (while routinely ignoring arson and vandalism at crisis pregnancy centers chargeable under the same law) is hard to deny. But the Department’s meritless criminal prosecution of Dr. Eithan Haim, a whistleblowing physician who exposed covert gender medicine procedures inflicted on minors at Texas Children’s Hospital—performed in violation of state law—is an outrageous abuse of its law enforcement authority. The principal wrongdoer in Dr. Haim’s case—leading the charge in his criminal prosecution for alleged violation of the Health Insurance Portability and Accountability Act (“HIPAA”)—is Assistant United States Attorney Tina Ansari, a Democratic donor who targeted Haim in part during a time when her law lic

In [5]:
# Define a task function
def summarize(text):
    outputs = pipe(text, max_length=150, clean_up_tokenization_spaces=True)
    return outputs[0]['summary_text']

In [6]:
# Pass the prompt into the task function
summarize(df['text'][0])

'Dr. Eithan Haim exposed covert gender medicine procedures at Texas Children’s Hospital. Haim knew the hospital “secretly continued to perform transgender medical interventions, including the use of implantable puberty blockers, on minor children” Haim did not disclose the name of any patients, or any personally identifiable medical information protected by federal law.'

In [10]:
for i in df['text'][47:89]:
    summary = summarize(i)
    summaries.append(summary)

RuntimeError: MPS backend out of memory (MPS allocated: 17.46 GB, other allocations: 23.11 MB, max allowed: 18.13 GB). Tried to allocate 1.25 GB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [12]:
len(summaries)

47

In [21]:
summarize(df['text'][2])

'The U.S. Supreme Court heard oral arguments in one of the term’s marquee cases: United States v. Skrmetti. At issue is the constitutionality of a Tennessee law, SB1 (codified at\xa0Tenn. Code Ann. § 68-33-103(a)(1), which prohibits any medical procedure for the purpose of “Enabling a minor to identify with, or live as, a purported identity inconsistent with the minor’S sex”'

In [22]:
summarize(df['text'][3])

'President Donald Trump has issued scores of executive orders related to gender identity. Julian Zelizer: Critics claim that he lacks the authority to do so. He says Congress can legislate through appropriations laws but only “as long as it does so clearly” Zelizer says the president’s executive order does not violate any specific law.'

In [27]:
import torch
import gc

# Clear all caches
torch.mps.empty_cache()
gc.collect()

# Restart your kernel/notebook

8008